# структурные сигналы: локация, фильтры, память, микрокатегория

BM25 нашел правильное объявление в топ-50 у 1/3 запросов, а в топ-200 у 1/2. ченкул анализ промахов и понял, что текст часто не различается: по запросу репетитор по химии 10 одинаково подходящих объявлений, а выбрано 1. Здесь проверяем, какие нетекстовые признаки помогают поднять выбранное наверх. каждый сигнал меряем одинаково переупорядочиваем топ-200 bm25 и смотрим recall@50

In [1]:
import sys, time
sys.path.insert(0, "..")

from pathlib import Path
import numpy as np
import polars as pl

from src.params import parse_item, parse_query
from src.text import normalize
from src.metrics import recall_at_k, recall_by_slice

PROC = Path("../data/processed")
val = pl.read_parquet(PROC / "val_queries.parquet")
corpus = pl.read_parquet(PROC / "val_corpus.parquet")
bm25_idx = np.load(PROC / "cands" / "bm25_val_top200.npy")

item_ids = corpus["item_id"].to_list()
relevant = {q: set(r) for q, r in zip(val["query_id"], val["relevant"])}
is_seen = dict(zip(val["query_id"], val["is_seen"]))

def evaluate(rows, label=""):
    pred = {q: [item_ids[j] for j in row] for q, row in zip(val["query_id"], rows)}
    r50 = recall_at_k(pred, relevant, 50)
    by_seen = recall_by_slice(pred, relevant, is_seen, 50)
    print(label, "-> recall@50:", round(r50, 4), "  (unseen", round(by_seen[False][0], 3), " seen", round(by_seen[True][0], 3), ")")
    return pred

def bring_forward(rows, mask_fn):
    # объявления, для которых mask_fn вернула true, ставим вперед, порядок внутри групп не трогаем
    out = []
    for i, row in enumerate(rows):
        m = mask_fn(i, row)
        out.append(np.concatenate([row[m], row[~m]]))
    return out

_ = evaluate(bm25_idx, "bm25 как есть")

bm25 как есть -> recall@50: 0.3394   (unseen 0.376  seen 0.277 )


## разбирает локацию

сначала разбираемся, что такое location_id: город или что-то крупнее. считаем разброс координат объявлений внутри одного id

In [2]:
loc_stats = (corpus.filter(pl.col("item_latitude").is_not_null()).with_columns(pl.col("item_latitude").cast(pl.Float64), pl.col("item_longitude").cast(pl.Float64)).group_by("item_location_id").agg(pl.len().alias("n"), pl.col("item_latitude").std().alias("lat_std"), pl.col("item_longitude").std().alias("lon_std")).filter(pl.col("n") >= 20))

loc_stats = loc_stats.with_columns((pl.col("lat_std") * 111).alias("spread_km"))
print("локаций с 20+ объявлениями", len(loc_stats))
print("разброс координат внутри локации, км медиана", round(loc_stats["spread_km"].median(), 1), " 90-й процентиль", round(loc_stats["spread_km"].quantile(0.9), 1))
print("самые крупные по числу объявлений")
print(loc_stats.sort("n", descending=True).head(5).select("item_location_id", "n", "spread_km"))

локаций с 20+ объявлениями 714
разброс координат внутри локации, км медиана 1.3  90-й процентиль 4.8
самые крупные по числу объявлений
shape: (5, 3)
┌──────────────────┬───────┬────────────┐
│ item_location_id ┆ n     ┆ spread_km  │
│ ---              ┆ ---   ┆ ---        │
│ i64              ┆ u32   ┆ f64        │
╞══════════════════╪═══════╪════════════╡
│ 637640           ┆ 19860 ┆ 107.853801 │
│ 653240           ┆ 12319 ┆ 198.870197 │
│ 633540           ┆ 4563  ┆ 3.728888   │
│ 650400           ┆ 3721  ┆ 143.63387  │
│ 641780           ┆ 3240  ┆ 151.812978 │
└──────────────────┴───────┴────────────┘


In [3]:
item_loc = corpus["item_location_id"].to_numpy()
q_loc = val["search_location_id"].to_numpy()
loc_by_id = dict(zip(corpus["item_id"], corpus["item_location_id"]))

share = np.mean([loc_by_id[i] == l for rel, l in zip(val["relevant"], q_loc) for i in rel])
print("позитивов в локации запроса", round(share, 3))

same_loc_first = bring_forward(bm25_idx, lambda i, row: item_loc[row] == q_loc[i])
_ = evaluate(same_loc_first)

позитивов в локации запроса 0.785
 -> recall@50: 0.4964   (unseen 0.506  seen 0.48 )


Разброс внутри локации единицы км, значит id это город а не регион, и совпадение по нему говорит много. позитив в той же локации у 79% запросов, а перестановка "свои вперед" дает самый большой прирост из всего, что видел


In [4]:
delivery = val["search_is_delivery_search"].to_numpy()
same = np.array([all(loc_by_id[i] == l for i in rel) for rel, l in zip(val["relevant"], q_loc)])
print("поисков с доставкой:", int(delivery.sum()), "из", len(val))
print("доля позитивов не в своей локации: без доставки", round(1 - same[delivery == 0].mean(), 3), " с доставкой", round(1 - same[delivery == 1].mean(), 3) if delivery.sum() else "нет таких")

поисков с доставкой: 0 из 2500
доля позитивов не в своей локации: без доставки 0.223  с доставкой нет таких


## Фильтры вида и типа услуги

Если в запросе стоит "Вид услуги Красота, здоровье"  пользователь видел только объявления с таким видом. проверяем, правда ли это по позитивам, и что дает перестановка.

In [5]:
q_parsed = [parse_query(p) for p in val["search_infm_params_text"]]
i_parsed = [parse_item(p) for p in corpus["item_infm_params_text"]]
item_vid = np.array([normalize(p["vid"]) for p in i_parsed])
item_tip = np.array([normalize(p["tip"]) for p in i_parsed])
vid_by_id = dict(zip(corpus["item_id"], item_vid))
tip_by_id = dict(zip(corpus["item_id"], item_tip))

with_vid = [(qp, rel) for qp, rel in zip(q_parsed, val["relevant"]) if qp["vid"]]
match = [vid_by_id[i] == normalize(qp["vid"]) for qp, rel in with_vid for i in rel]
print("запросов с фильтром вида:", len(with_vid), " позитив совпадает по виду:", round(np.mean(match), 3))

with_tip = [(qp, rel) for qp, rel in zip(q_parsed, val["relevant"]) if qp["tip"]]
match = [tip_by_id[i] == normalize(qp["tip"]) for qp, rel in with_tip for i in rel]
print("запросов с фильтром типа:", len(with_tip), " позитив совпадает по типу:", round(np.mean(match), 3))

def filter_ok(i, row):
    qp = q_parsed[i]
    ok = np.ones(len(row), dtype=bool)
    if qp["vid"]:
        ok &= item_vid[row] == normalize(qp["vid"])
    if qp["tip"]:
        ok &= item_tip[row] == normalize(qp["tip"])
    return ok

filter_first = bring_forward(bm25_idx, filter_ok)
_ = evaluate(filter_first, "совпадение фильтра вперед")
both = bring_forward(same_loc_first, filter_ok)
both = bring_forward(both, lambda i, row: item_loc[row] == q_loc[i])
_ = evaluate(both, "фильтр, потом локация")

запросов с фильтром вида: 1272  позитив совпадает по виду: 0.983
запросов с фильтром типа: 639  позитив совпадает по типу: 0.809


совпадение фильтра вперед -> recall@50: 0.3538   (unseen 0.388  seen 0.295 )


фильтр, потом локация -> recall@50: 0.5005   (unseen 0.511  seen 0.483 )


## Память по запросу

Для запросов, чей текст уже встречался в трейне: что тогда выбирали, и есть ли оно в корпусе

In [6]:
train_rest = pl.scan_parquet(PROC / "train_rest.parquet").select("search_query", "search_location_id", "item_id").collect()
train_rest = train_rest.with_columns(pl.col("search_query").map_elements(normalize, return_dtype=pl.String).alias("q_norm"))
memory = dict(train_rest.group_by("q_norm").agg(pl.col("item_id").unique()).iter_rows())
memory_loc = dict(train_rest.group_by("q_norm", "search_location_id").agg(pl.col("item_id").unique()) .select(pl.concat_list("q_norm", "search_location_id").cast(pl.List(pl.String)).list.join("|"), "item_id").iter_rows())
in_corpus = set(item_ids)
pos_in_corpus = {i: j for j, i in enumerate(item_ids)}

hits_any = 0
hits_loc = 0
mem_rows = []
for i, (q, l, row) in enumerate(zip(val["search_query"], q_loc, bm25_idx)):
    qn = normalize(q)
    remembered = [x for x in memory.get(qn, []) if x in in_corpus]
    remembered_loc = [x for x in memory_loc.get(qn + "|" + str(l), []) if x in in_corpus]
    qid = val["query_id"][i]
    if set(remembered) & relevant[qid]: hits_any += 1
    if set(remembered_loc) & relevant[qid]: hits_loc += 1
    # сначала то, что помним по этому тексту в этой локации, потом остальное из памяти, потом bm25
    front = [pos_in_corpus[x] for x in remembered_loc] + [pos_in_corpus[x] for x in remembered if x not in remembered_loc]
    rest = [j for j in row if j not in set(front)]
    mem_rows.append(np.array(front + rest)[:200])

print("запросов, где память по тексту содержит правильный ответ:", hits_any, " по тексту и локации:", hits_loc, " из", int(val["is_seen"].sum()), "seen")
_ = evaluate(mem_rows, "память вперед, потом bm25")

запросов, где память по тексту содержит правильный ответ: 0  по тексту и локации: 0  из 925 seen
память вперед, потом bm25 -> recall@50: 0.3325   (unseen 0.376  seen 0.258 )


## Микрокатегория

У запроса ее нет но по трейну можно выучить, в какой микрокатегории обычно выбирают. классификатор - линейный над символьными n-граммами текста запроса. он быстро и вполне хорошо должен работать на коротких строках

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

labeled = (pl.scan_parquet(PROC / "train_rest.parquet") .select("search_query", "item_microcat_id").collect())
X_text = [normalize(q) for q in labeled["search_query"]]
y = labeled["item_microcat_id"].to_numpy()
print("обучающих строк", len(X_text), " микрокатегорий:", len(set(y)))

t = time.time()
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=400_000)
X = vec.fit_transform(X_text)
clf = SGDClassifier(loss="log_loss", alpha=2e-7, max_iter=8, tol=None, n_jobs=-1, random_state=0)
clf.fit(X, y)
print("обучили за", round((time.time() - t) / 60, 1), "минут")

# проверяем на валидации: попала ли микрокатегория выбранного объявления в топ предсказаний
microcat_by_id = dict(zip(corpus["item_id"], corpus["item_microcat_id"]))
Xv = vec.transform([normalize(q) for q in val["search_query"]])
proba = clf.predict_proba(Xv)
classes = clf.classes_
top3 = classes[np.argsort(-proba, axis=1)[:, :3]]
for k in [1, 3]:
    hit = [any(microcat_by_id[i] in top3[n, :k] for i in rel) for n, rel in enumerate(val["relevant"])]
    print("микрокатегория позитива в топ-" + str(k) + " предсказаний:", round(np.mean(hit), 3))

обучающих строк 492026  микрокатегорий: 212


обучили за 0.6 минут
микрокатегория позитива в топ-1 предсказаний: 0.725
микрокатегория позитива в топ-3 предсказаний: 0.888


In [8]:
item_mc = corpus["item_microcat_id"].to_numpy()
mc_first = bring_forward(bm25_idx, lambda i, row: np.isin(item_mc[row], top3[i]))
_ = evaluate(mc_first, "микрокатегория из топ-3 вперед")

микрокатегория из топ-3 вперед -> recall@50: 0.3418   (unseen 0.369  seen 0.296 )


## все вместе

собираем сигналы в один скор поверх BM25 и подбираем веса перебором. это еще не реранкер, а грубый аналог: он покажет, какой recall можно снять с кандидатов BM25 без обучения

In [9]:
bm25_rank = np.tile(np.arange(200), (len(val), 1))
loc_match = np.stack([item_loc[row] == q_loc[i] for i, row in enumerate(bm25_idx)])
flt_match = np.stack([filter_ok(i, row) for i, row in enumerate(bm25_idx)])
mc_match = np.stack([np.isin(item_mc[row], top3[i]) for i, row in enumerate(bm25_idx)])
# вероятность микрокатегории объявления по классификатору; если такой микрокатегории он не знает, ставим 0
class_pos = {c: j for j, c in enumerate(classes)}
mc_prob = np.stack([np.array([proba[i][class_pos[m]] if m in class_pos else 0.0 for m in item_mc[row]]) for i, row in enumerate(bm25_idx)])

best = None
for w_loc in [0, 20, 50, 100]:
    for w_flt in [0, 20, 50, 100]:
        for w_mc in [0, 20, 50]:
            # чем меньше скор, тем выше: ранг bm25 минус бонусы
            score = bm25_rank - w_loc * loc_match - w_flt * flt_match - w_mc * mc_prob
            rows = [row[np.argsort(s, kind="stable")] for row, s in zip(bm25_idx, score)]
            pred = {q: [item_ids[j] for j in row] for q, row in zip(val["query_id"], rows)}
            r = recall_at_k(pred, relevant, 50)
            if best is None or r > best[0]:
                best = (r, w_loc, w_flt, w_mc)
print("лучшая комбинация: recall@50 =", round(best[0], 4), " веса: локация", best[1], " фильтр", best[2], " микрокатегория", best[3])

score = bm25_rank - best[1] * loc_match - best[2] * flt_match - best[3] * mc_prob
rows = [row[np.argsort(s, kind="stable")] for row, s in zip(bm25_idx, score)]
_ = evaluate(rows, "bm25 + все сигналы")
np.save(PROC / "cands" / "bm25_structured_val_top200.npy", np.stack(rows))

лучшая комбинация: recall@50 = 0.4729  веса: локация 100  фильтр 100  микрокатегория 50
bm25 + все сигналы -> recall@50: 0.4729   (unseen 0.489  seen 0.445 )
